# 02 - Localize Refusal Features

Runs the 5 strategies (neuronpedia search, linear probe, contrastive diff, attribution, rank fusion) across candidate layers. Selects best layer.

**Critical:** if AUROC < 0.85 in all layers, activate plan B (train own SAE, try Llama, etc.).

In [ ]:
import os
import sys
REPO_DIR = '/kaggle/working/pc-sae'
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

from src.utils import setup_hf_auth
setup_hf_auth()

In [ ]:
!python scripts/02_localize_features.py

In [ ]:
import json
import matplotlib.pyplot as plt
from src.utils import METRICS, CANDIDATE_LAYERS, load_json

decision = load_json(METRICS / 'localization_decision.json')
print('=== DECISION ===')
print(f"best layer: {decision['best_layer']}")
print(f"best AUROC: {decision['best_auroc']:.4f}")
print(f"n features selected: {len(decision['selected_feature_indices'])}")
print('\n=== PER LAYER ===')
for L, info in decision['summary_per_layer'].items():
    print(f"  layer {L}: AUROC={info['auroc']:.4f} nonzero={info['nonzero']}")

In [ ]:
reports = {}
for L in CANDIDATE_LAYERS:
    f = METRICS / f'localization_layer_{L}.json'
    if f.exists():
        reports[L] = load_json(f)

layers = sorted(reports.keys())
aurocs = [reports[L]['probe_auroc'] for L in layers]
nonzeros = [reports[L]['probe_nonzero'] for L in layers]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar([str(L) for L in layers], aurocs, color='steelblue')
ax[0].axhline(0.85, color='r', linestyle='--', label='min acceptable')
ax[0].axhline(0.95, color='g', linestyle='--', label='target')
ax[0].set_xlabel('layer'); ax[0].set_ylabel('AUROC'); ax[0].set_title('Linear Probe AUROC')
ax[0].legend(); ax[0].set_ylim(0, 1)
ax[1].bar([str(L) for L in layers], nonzeros, color='orange')
ax[1].set_xlabel('layer'); ax[1].set_ylabel('nonzero coefs'); ax[1].set_title('Probe Sparsity')
plt.tight_layout(); plt.savefig('outputs/results/figures/layer_comparison.png', dpi=100)
plt.show()

In [ ]:
best = decision['best_layer']
r = reports[best]
print(f'=== LAYER {best} TOP FEATURES ===\n')
print('Top 15 by linear probe (idx, coef):')
for f in r['probe_top'][:15]:
    print(f'  {f[0]}: coef={f[1]:+.3f}')
print('\nTop 15 by contrastive diff:')
for f in r['contrastive_top'][:15]:
    print(f'  {f[0]}: diff={f[1]:+.3f}')
print('\nTop 15 by attribution (cohen d):')
for f in r['attribution_top'][:15]:
    print(f'  {f[0]}: d={f[1]:+.3f}')

In [ ]:
in_probe = {f[0] for f in r['probe_top'][:50]}
in_contrast = {f[0] for f in r['contrastive_top'][:50]}
in_attr = {f[0] for f in r['attribution_top'][:50]}
intersection = in_probe & in_contrast & in_attr
print('EXPLORATORY view: strict 3-way intersection (top-50).')
print('NOT the downstream selection: the PC is trained on rank_fusion(top_k=128),')
print('a count-weighted UNION of the rankings, not this intersection.')
print(f'features in all 3 rankings (top-50): {len(intersection)}')
print(sorted(intersection))
print(f'actual selection (rank_fusion union): {len(decision["selected_feature_indices"])} features')

In [ ]:
np_results = r.get('neuronpedia', {})
for kw, feats in np_results.items():
    if feats:
        print(f'\n{kw}:')
        for f in feats[:5]:
            print(f"  {f['index']}: {f['description'][:100]}")

In [ ]:
!cat outputs/logs/localize.log 2>/dev/null | tail -40
print('\n--- ERRORS ---')
!cat outputs/logs/errors.log 2>/dev/null | tail -20